### Подготовка датасета по показателю объем убоя КРС

In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import kpss
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from statsmodels.graphics.tsaplots import plot_acf
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

from pylab import rcParams
from IPython.display import display
import math
from prophet import Prophet
pd.set_option('display.max_columns', 130)


import warnings
from statsmodels.tools.sm_exceptions import InterpolationWarning
warnings.simplefilter("ignore", category=InterpolationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)



In [53]:
df = pd.read_excel("../../Data cleansing/output data/Просуммированные по категориям с доп регрессорами.xlsx")
df.head(5)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
0,Верблюды,АКМОЛИНСКАЯ ОБЛАСТЬ,0.00,0.00,0.40,0.00,0.00,0.00,0.09,1.00,0.00,0.00,0.20,0.00,0.00,0.18,0.28,0.00,0.00,0.40,0.00,0.00,0.65,0.00,0.31,1.00,0.00,0.00,0.00,0.00,0.00,2.01,0.00,1.20,0.00,0.00,2.18,0.39,0.00,0.00,1.04,0.00,0.14,2.08,0.00,0.00,0.00,0.00,0.00,0.30,0.00,0.00,0.00,0.66,0.00,0.33,0.00,0.9,0.00,0.00,0.00,10.08,0.00,0.0,0.00,0.0,0.00,0.54,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.36,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.40,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.0,0.0,0.00,0.0,0.00,0.0,0.00,0.00,0.0,0.00
1,Верблюды,АКТЮБИНСКАЯ ОБЛАСТЬ,101.98,67.47,374.84,115.59,218.72,14.15,19.77,3.00,39.16,46.16,238.56,463.35,109.04,72.94,384.96,114.35,221.89,10.93,18.79,3.50,38.53,46.64,216.08,472.07,110.32,68.17,380.15,127.12,217.61,14.98,21.26,7.14,44.63,51.78,239.43,513.33,115.70,69.40,371.53,130.17,218.41,14.59,21.76,7.19,45.97,55.48,252.68,527.66,117.94,70.37,385.75,117.80,218.45,15.54,21.78,7.2,47.06,58.02,257.54,543.35,119.80,71.3,396.90,121.8,228.10,15.90,21.9,7.2,63.00,59.90,260.20,552.10,121.60,71.8,398.40,128.50,228.80,13.10,22.50,7.20,63.90,61.90,150.70,554.90,121.30,73.3,404.20,128.70,234.90,13.60,23.60,6.3,49.80,62.4,153.70,552.8,118.07,69.17,384.21,121.01,223.46,11.76,21.39,5.52,48.18,61.39,150.87,531.79,119.50,72.60,391.0,126.1,229.10,12.2,24.10,5.7,49.87,63.00,154.3,538.40
2,Верблюды,АЛМАТИНСКАЯ ОБЛАСТЬ,1.00,0.20,51.60,25.40,0.00,61.90,76.87,95.57,16.09,0.10,10.80,46.78,15.06,13.70,131.80,9.20,17.89,130.12,2.00,0.00,4.99,2.06,4.90,44.48,15.06,23.94,41.60,19.54,3.00,15.14,18.50,14.82,14.52,12.60,9.92,43.11,20.72,2.96,15.24,0.00,1.00,20.03,3.44,7.77,117.25,1.00,0.00,20.34,21.95,3.04,17.91,26.81,9.16,18.80,13.37,0.0,17.29,15.90,4.69,23.10,10.85,9.7,42.75,0.0,6.21,21.10,4.0,2.6,28.10,4.53,79.50,93.18,23.70,11.2,16.00,2.00,3.10,19.62,2.30,4.85,11.18,2.80,9.65,36.44,10.60,11.5,26.10,7.20,12.75,11.20,11.20,27.0,25.26,17.9,21.90,12.5,16.55,16.81,22.78,12.10,12.44,6.85,39.65,17.22,8.85,27.48,0.48,29.72,18.90,17.40,12.5,16.4,11.60,16.7,15.70,24.0,7.70,6.90,5.4,12.60
3,Верблюды,АТЫРАУСКАЯ ОБЛАСТЬ,213.89,167.70,306.60,164.97,342.57,192.00,43.60,113.03,262.37,193.97,308.17,1087.33,325.10,190.60,301.10,154.84,328.50,220.30,66.10,118.20,234.90,196.51,270.92,974.18,303.30,167.41,323.54,182.34,349.10,249.40,57.57,88.40,221.21,199.63,278.80,964.02,305.22,159.62,326.87,159.50,367.20,257.50,60.01,78.10,253.27,332.00,355.05,989.86,280.08,165.05,334.51,134.85,367.97,321.22,126.22,93.7,263.10,366.66,348.54,1041.38,293.40,154.8,339.44,143.6,405.80,308.35,88.5,130.5,575.30,458.70,357.62,998.10,292.76,188.3,487.02,129.28,421.44,351.20,95.77,97.54,624.80,631.65,499.30,1179.66,286.67,189.8,513.16,171.46,402.10,432.22,121.53,126.2,637.50,651.7,482.94,1412.2,307.85,200.59,466.60,182.63,414.56,405.19,184.80,154.96,717.92,719.01,522.83,1164.77,323.81,423.76,441.9,219.1,487.69,446.9,191.43,133.5,730.34,709.68,537.6,930.28
4,Верблюды,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,0.30,0.00,0.00,1.14,6.12,4.70

In [54]:
df_krs = df[df['Показатель'].isin(['КРС', 'Температура', 'Осадки', 'Поголовье: КРС', 'Цена: Говядина'])]
df_krs.sample(10)

,Показатель,Регион,2015-01,2015-02,2015-03,2015-04,2015-05,2015-06,2015-07,2015-08,2015-09,2015-10,2015-11,2015-12,2016-01,2016-02,2016-03,2016-04,2016-05,2016-06,2016-07,2016-08,2016-09,2016-10,2016-11,2016-12,2017-01,2017-02,2017-03,2017-04,2017-05,2017-06,2017-07,2017-08,2017-09,2017-10,2017-11,2017-12,2018-01,2018-02,2018-03,2018-04,2018-05,2018-06,2018-07,2018-08,2018-09,2018-10,2018-11,2018-12,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,2019-09,2019-10,2019-11,2019-12,2020-01,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12
203,Поголовье: КРС,ГШЫМКЕНТ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.088600e+04,7.613100e+04,75339.000000,74160.000000,74368.000000,73776.000000,74246.000000,72529.000000,7.185000e+04,7.236400e+04,7.077700e+04,7.530800e+04,7.345400e+04,8.611800e+04,8.516600e+04,8.451700e+04,83226.000000,82415.000000,8.074100e+04,8.207100e+04,7.427800e+04,7.371500e+04,7.486400e+04,7.519100e+04,8.258400e+04,8.272500e+04,8.169800e+04,8.157400e+04,8.095800e+04,7.894800e+04,7.846900e+04,7.622300e+04,7.442500e+04,7.373100e+04,7.446600e+04,7.547200e+04,8.244400e+04,8.143500e+04,8.040200e+04,7.922600e+04,7.857400e+04,7.790000e+04,7.763500e+04,7.522400e+04,7.358800e+04,7.191400e+04,7.226800e+04,7.276700e+04,7.979500e+04,7.986000e+04,7.880200e+04,7.771100e+04,7.775800e+04,7.705900e+04,7.631200e+04,4103.000000,4382.000000,4358.000000,4394.000000,4514.000000,4585.000000,4578.000000,4867.000000,5018.000000,3956.000000,5656.000000,5974.000000,94841.000000,91980.000000,90275.000000,89035.000000,89502.000000,88345.000000,1.102320e+05,1.094440e+05,1.095570e+05,1.085920e+05,1.074790e+05,1.065180e+05
41,КРС,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.079520e+03,8.805580e+03,7908.160000,9309.420000,8758.150000,10167.940000,13879.820000,7920.240000,7.586870e+03,9.335520e+03,7.608450e+03,8.279780e+03,1.055346e+04,7.745530e+03,7.067920e+03,9.366440e+03,13418.880000,11198.800000,1.584204e+04,8.100430e+03,7.896170e+03,1.012037e+04,7.816520e+03,8.071890e+03,1.019411e+04,7.587560e+03,8.019090e+03,1.037002e+04,1.331023e+04,1.333519e+04,1.566702e+04,8.540110e+03,8.646890e+03,8.820660e+03,8.933600e+03,8.980520e+03,7.566120e+03,7.968020e+03,1.246794e+04,1.098813e+04,1.279660e+04,1.310092e+04,1.756676e+04,8.683130e+03,8.849710e+03,9.935200e+03,8.989710e+03,9.127380e+03,7.984620e+03,8.620210e+03,1.206302e+04,1.188536e+04,1.407690e+04,1.295565e+04,1.750862e+04,6439.160000,6467.490000,7094.660000,8814.220000,7313.340000,6093.710000,6506.790000,10592.780000,11356.820000,8339.580000,9039.420000,14533.010000,6351.970000,11020.910000,7050.970000,9224.010000,6776.160000,15715.540000,6.948860e+03,9.372790e+03,1.573807e+04,1.056650e+04,1.144198e+04,2.935324e+04
204,Поголовье: КРС,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.163656e+06,1.020139e+06,998103.000000,972607.000000,948020.000000,921291.000000,994197.000000,993585.000000,1.036129e+06,1.105563e+06,1.151679e+06,1.175596e+06,1.157862e+06,1.070131e+06,1.050766e+06,1.024687e+06,989405.000000,960061.000000,1.052929e+06,1.052923e+06,1.053058e+06,1.095066e+06,1.166149e+06,1.229438e+06,1.257755e+06,1.229017e+06,1.130048e+06,1.109185e+06,1.080921e+06,1.047005e+06,1.011931

In [55]:
# Step 1: Pivot to wide format (each indicator becomes columns of periods)
df_wide = df_krs.pivot(index="Регион", columns="Показатель")

# Step 2: Flatten multi-level columns: ('2015-01', 'КРС') → 'КРС_2015-01'
df_wide.columns = [f"{col[1]}_{col[0]}" for col in df_wide.columns]
df_wide = df_wide.reset_index()

# Step 3: Melt: one row per region-period-indicator
df_melted = df_wide.melt(id_vars="Регион", var_name="indicator_period", value_name="value")

# Step 4: Extract 'Период' and 'Показатель' from the combined column
df_melted["Период"] = df_melted["indicator_period"].str.extract(r"_(\d{4}-\d{2})$")
df_melted["Показатель"] = df_melted["indicator_period"].str.extract(r"^(.+)_\d{4}-\d{2}")

# Step 5: Pivot again to get final modeling format: one row per region+period, one column per indicator
df_krs = df_melted.pivot_table(index=["Регион", "Период"], columns="Показатель", values="value").reset_index()
print(df_krs.groupby("Регион").size().reset_index(name="Количество строк"))
df_krs

                            Регион  Количество строк
0              АКМОЛИНСКАЯ ОБЛАСТЬ               120
1              АКТЮБИНСКАЯ ОБЛАСТЬ               120
2              АЛМАТИНСКАЯ ОБЛАСТЬ               120
3               АТЫРАУСКАЯ ОБЛАСТЬ               120
4   ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               120
5                          ГАЛМАТЫ               120
6                          ГАСТАНА               120
7                         ГШЫМКЕНТ                79
8               ЖАМБЫЛСКАЯ ОБЛАСТЬ               120
9    ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ               120
10          КАРАГАНДИНСКАЯ ОБЛАСТЬ               120
11            КОСТАНАЙСКАЯ ОБЛАСТЬ               120
12          КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ               120
13           МАНГИСТАУСКАЯ ОБЛАСТЬ               120
14                    ОБЛАСТЬ АБАЙ                31
15                  ОБЛАСТЬ ЖЕТІСУ                31
16                  ОБЛАСТЬ ҰЛЫТАУ                31
17            ПАВЛОДАРСКАЯ ОБЛАСТЬ            

Показатель,Регион,Период,КРС,Осадки,Поголовье: КРС,Температура,Цена: Говядина
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,4455.35,9.8,372560.0,-12.490323,100.000000
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,3654.20,9.8,399442.0,-10.192857,100.000000
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,4287.08,8.3,425605.0,-5.870968,100.000000
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,3923.21,8.8,440023.0,4.490000,99.800000
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,3849.70,42.8,444647.0,14.574194,99.800000
...,...,...,...,...,...,...,...
2166,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-08,9372.79,0.0,1120067.0,27.874194,164.973838
2167,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-09,15738.07,0.5,1101103.0,20.766667,164.973838
2168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-10,10566.50,13.6,1078583.0,13.200000,165.138812
2169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-11,11441.98,12.1,1056289.0,6.500000,165.469090


In [56]:
df_krs = df_krs[df_krs["Регион"] != 'РЕСПУБЛИКА КАЗАХСТАН']

In [57]:
df_krs

Показатель,Регион,Период,КРС,Осадки,Поголовье: КРС,Температура,Цена: Говядина
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-01,4455.35,9.8,372560.0,-12.490323,100.000000
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-02,3654.20,9.8,399442.0,-10.192857,100.000000
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-03,4287.08,8.3,425605.0,-5.870968,100.000000
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-04,3923.21,8.8,440023.0,4.490000,99.800000
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2015-05,3849.70,42.8,444647.0,14.574194,99.800000
...,...,...,...,...,...,...,...
2166,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-08,9372.79,0.0,1120067.0,27.874194,164.973838
2167,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-09,15738.07,0.5,1101103.0,20.766667,164.973838
2168,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-10,10566.50,13.6,1078583.0,13.200000,165.138812
2169,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2024-11,11441.98,12.1,1056289.0,6.500000,165.469090


In [58]:
df_krs.to_excel("Датасет по КРС с регрессорами.xlsx", index=False)

In [59]:
df_krs = df_krs.drop(columns=['Осадки', 'Поголовье: КРС', 'Температура',
       'Цена: Говядина'])
df_krs.sample(10)

Показатель,Регион,Период,КРС
1690,ОБЛАСТЬ ЖЕТІСУ,2024-02,1650.36
1669,ОБЛАСТЬ АБАЙ,2024-12,18068.25
1394,КОСТАНАЙСКАЯ ОБЛАСТЬ,2024-08,2421.12
1729,ОБЛАСТЬ ҰЛЫТАУ,2024-10,507.00
2024,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2019-05,2675.35
48,АКМОЛИНСКАЯ ОБЛАСТЬ,2019-01,4842.70
1541,МАНГИСТАУСКАЯ ОБЛАСТЬ,2016-11,68.25
676,ГАЛМАТЫ,2021-05,1.00
1805,ПАВЛОДАРСКАЯ ОБЛАСТЬ,2021-02,3169.99
1627,МАНГИСТАУСКАЯ ОБЛАСТЬ,2024-01,95.22


In [60]:
df_krs.to_excel("Датасет по КРС.xlsx", index=False)

In [61]:
# 1) приведение типов
df = df_krs.copy()
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m", errors="coerce")
df["КРС"] = pd.to_numeric(df["КРС"], errors="coerce")

# 2) wide-таблица: строки — месяцы, столбцы — регионы, значения — КРС
df_krs_wide = (df
        .pivot_table(index="Период", columns="Регион", values="КРС", aggfunc="sum")  # если дублей нет — можно aggfunc="first"
        .sort_index())
# 3) Убираем лишний уровень индекса у колонок
df_krs_wide.columns.name = None

# 4) Возвращаем "Период" в строковый формат YYYY-MM
df_krs_wide = df_krs_wide.reset_index()
df_krs_wide["Период"] = df_krs_wide["Период"].dt.strftime("%Y-%m")

df_krs_wide.to_excel("Датасет по КРС wide.xlsx", index=False)
df_krs_wide

,Период,АКМОЛИНСКАЯ ОБЛАСТЬ,АКТЮБИНСКАЯ ОБЛАСТЬ,АЛМАТИНСКАЯ ОБЛАСТЬ,АТЫРАУСКАЯ ОБЛАСТЬ,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,ГАЛМАТЫ,ГАСТАНА,ГШЫМКЕНТ,ЖАМБЫЛСКАЯ ОБЛАСТЬ,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,КАРАГАНДИНСКАЯ ОБЛАСТЬ,КОСТАНАЙСКАЯ ОБЛАСТЬ,КЫЗЫЛОРДИНСКАЯ ОБЛАСТЬ,МАНГИСТАУСКАЯ ОБЛАСТЬ,ОБЛАСТЬ АБАЙ,ОБЛАСТЬ ЖЕТІСУ,ОБЛАСТЬ ҰЛЫТАУ,ПАВЛОДАРСКАЯ ОБЛАСТЬ,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,ТУРКЕСТАНСКАЯ ОБЛАСТЬ
0,2015-01,4455.35,5786.80,6087.47,1957.38,4151.27,35.38,7.48,NaN,2731.22,2406.30,4276.61,3858.83,1446.34,30.15,NaN,NaN,NaN,2646.74,6302.91,NaN
1,2015-02,3654.20,5425.85,4454.61,1755.18,6473.64,14.65,27.08,NaN,3109.06,2980.56,2663.85,5661.27,1165.32,225.79,NaN,NaN,NaN,2934.07,3236.66,NaN
2,2015-03,4287.08,6578.37,16005.48,2085.52,6837.39,34.19,13.91,NaN,2538.81,3497.07,3105.64,4313.48,1233.92,581.50,NaN,NaN,NaN,3230.62,1531.42,NaN
3,2015-04,3923.21,5130.34,3934.04,1346.93,5875.35,40.53,19.11,NaN,3379.88,3061.07,2216.56,4841.38,1355.81,574.71,NaN,NaN,NaN,2731.05,2111.45,NaN
4,2015-05,3849.70,5668.00,7098.30,2073.99,5952.33,36.70,17.31,NaN,2921.49,3678.77,3436.05,7257.34,1189.73,270.72,NaN,NaN,NaN,2830.10,2218.93,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,2024-08,2412.70,4406.21,5453.39,1640.07,2845.28,2.30,1.70,352.82,4488.08,3873.69,3097.30,2421.12,1581.37,84.38,3114.29,2568.13,593.10,4061.35,3472.15,9372.79
116,2024-09,2909.66,5198.02,11705.42,2317.39,5000.88,12.50,3.40,350.02,4264.39,5466.61,5874.20,3723.42,1724.95,242.24,5453.19,7223.75,750.95,4014.27,5090.14,15738.07
117,2024-10,2608.27,3894.84,8880.02,2004.60,3396.22,4.00,2.60,466.45,3418.05,3586.35,2631.65,1925.96,1919.48,168.08,3460.45,6626.00,507.00,4697.04,2367.75,10566.50
118,2024-11,3649.19,4389.74,5882.78,3124.16,4578.81,5.90,2.00,494.05,5708.50,4578.05,4280.10,1575.93,1838.46,258.08,5847.30,6669.33,248.25,5946.29,2041.65,11441.98
